# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a worked example for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display basic metadata information
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")print(f"Published: {meta.datePublished}")print(f"License: {meta.license}")print(f"Authors: {getattr(meta, 'author', 'N/A')}")

## 2. Data Overview
Review the available record sets, their unique `@id`s, and associated fields.
We'll use the `record_sets` and `fields` registered in the Croissant metadata:
* **Record Set `@id`**: Unique identifier for each record set (table/logical dataset).
* **Fields**: Columns or variables within each record set, each with unique `@id`s.

In [ ]:
# Explore available record sets and fields using their @id
record_sets = list(dataset.record_sets)

print(f"Found {len(record_sets)} record set(s):\n")
for rset in record_sets:
    print(f"- Record Set @id: {rset['@id']}")
    print(f"  Name:       {rset.get('name', 'N/A')}")
    print(f"  Description:{rset.get('description', 'N/A')}")
    # List fields (columns)
    print("  Fields:")
    for field in rset.get('field', []):
        print(f"    - Field @id: {field.get('@id')}, name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')}")
    print("\n")    # Show a sample of first records for each record set, using @id
for rset in record_sets:
    rset_id = rset['@id']
    print(f"Example record(s) from record set @id={rset_id}:")
    try:
        records = list(dataset.records(record_set=rset_id))
        for i, rec in enumerate(records[:3]):
            print(f"  Record {i+1}: {rec}")
        if not records:
            print("  (No records found)")
    except Exception as e:
        print(f"  Could not load records: {e}")
    print()

## 3. Data Extraction
Let's load the data for each record set into a pandas DataFrame for further analysis.
You must always reference record sets and fields by their `@id` (as per Croissant).

Below we demonstrate this with all discovered record sets from the previous section:

In [ ]:
dataframes = {}
# List all record set @ids
record_set_ids = [rset['@id'] for rset in record_sets]

for rset_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rset_id))
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        print(f"Loaded DataFrame for record set @id: {rset_id} (shape={df.shape})")
        print(f"    Columns (@id): {df.columns.tolist()}")
    except Exception as ex:
        print(f"[Error] Could not load records for @id={rset_id}: {ex}")
        dataframes[rset_id] = None
    print()

# Choose the first record set as the demonstration target
if record_set_ids:
    test_rset = record_set_ids[0]
    if dataframes[test_rset] is not None:
        print(f"Top rows of primary record set (@id={test_rset}):")
        display(dataframes[test_rset].head())

## 4. Exploratory Data Analysis (EDA)
We'll apply some common data processing steps such as filtering, normalizing, and grouping, using Croissant `@id`s for all fields.

**Steps:**
1. Select a numeric field (by `@id`) for analysis. If needed, inspect the field types from the earlier overview.
2. Filter records based on this field using a chosen threshold.
3. Normalize the numeric field and show the result.
4. Optionally group by a categorical field (again, by `@id`).

In [ ]:
# --- EDA with explicit @id referencing ---
# Choose the primary record set for deeper analysis
record_set_id = record_set_ids[0] if record_set_ids else None
if not record_set_id:
    raise ValueError('No record set IDs found in metadata.')
df = dataframes[record_set_id]

print(f"Working with primary record set @id: {record_set_id}")

# Identify a numeric field from earlier overview or from df.dtypes.
print('Attempting to auto-detect a numeric field for EDA:')
numeric_field_id = None
for col in df.columns:
    # Try parsing as numeric
    try:
        pd.to_numeric(df[col])
        numeric_field_id = col
        if df[col].dtype.kind in 'ifc' or (df[col].dropna().apply(lambda x: str(x).replace('.', '', 1).isdigit()).all()):
            print(f"Using numeric field @id: {col}")
            break
    except:
        continue
if numeric_field_id is None:
    print('Warning: No numeric field auto-detected. Please examine columns above.')
else:
    # Convert the field to numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Set a threshold, e.g., the mean or a constant (10 as demonstration)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):\n")
    display(filtered_df.head())

    # Normalize the selected numeric field
    norm_field = f"{numeric_field_id}_normalized"
    filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_field]].head())

    # Attempt to group by a likely non-numeric field
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() < 12 and df[col].dtype == object:
            group_field_id = col
            break
    if group_field_id:
        print(f"Grouping by field @id: {group_field_id}\n")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        display(grouped_df.head())
    else:
        print('No suitable grouping field found.')

## 5. Visualization
Here we visualize the distribution of the selected numeric field and relationships to the group field (if present).

*Make sure all references are by `@id`.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(9, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True, color='dodgerblue')
    plt.xlabel(f'Numeric Variable (@id: {numeric_field_id})')
    plt.title(f'Distribution of {numeric_field_id} in record set {record_set_id}')
    plt.show()

    # If a group field was detected earlier, boxplot by group
    if group_field_id:
        plt.figure(figsize=(9,4))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} per {group_field_id} (@id)')
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, and analyze a dataset defined with the Croissant schema using `mlcroissant`, with strict use of `@id` references for all entities (record sets, fields, columns). You can now extend these methods for in-depth exploration and custom analytics on any Croissant-compatible FAIR dataset.

*Key observations:*
- Dataset metadata and structure accessed dynamically from schema URL
- Full traceability via Croissant `@id` usage
- Automated EDA and visualization ready for deeper domain analysis

For further analysis: consult the dataset's documentation and consider integrating more complex ML or statistical pipelines using these robust, reproducible access methods.